# Chemistry features and clustering

Owner: P3.

## Goal

Chemistry features and clustering

## Inputs

Recipe fields and split from P1; reference classifiers from P2

## Tasks

- [ ] Build UMF features, oxide groups and silica/alumina ratio.
- [ ] Run controlled ablation with and without additional chemistry features.
- [ ] Fit K-Means and hierarchical clustering without target labels.
- [ ] Plot PCA or UMAP and compare clusters to observed finish after fitting.

## Outputs

- `src/glazevision/features.py`
- `notebooks/03_features_clustering.ipynb`
- `reports/results.csv`

In [ ]:
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.feature_extraction import DictVectorizer
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from glazevision.data import joined_samples
from glazevision.features import recipe_features
from glazevision.experiments import run_tabular

train = joined_samples().query("split == 'train'").sample(n=500, random_state=42)
features = [recipe_features(row, True) for _, row in train.iterrows()]
matrix = StandardScaler().fit_transform(DictVectorizer(sparse=False).fit_transform(features))
for method in (KMeans(n_clusters=3, random_state=42), AgglomerativeClustering(n_clusters=3)):
    groups = method.fit_predict(matrix)
    display(train.assign(cluster=groups).groupby(["cluster", "surface"]).size().unstack(fill_value=0))
xy = PCA(n_components=2, random_state=42).fit_transform(matrix)
for label in train.surface.dropna().unique():
    mask = train.surface.eq(label).to_numpy()
    plt.scatter(xy[mask, 0], xy[mask, 1], label=label)
plt.legend()
plt.show()
run_tabular(names={"logistic"})
run_tabular(chemistry=True, names={"logistic"})

In [ ]:
# TODO(P3): Record actual findings, run settings and limitations after evaluation.